# Local LLM Clinical NER with Qwen2.5-14B-Instruct via Ollama

This notebook runs Qwen2.5 locally through Ollama, extracts clinical and PHI entities from a given input text, returns span-level JSON annotations, validates and repairs offsets, saves the output as JSON, and can later convert the extracted spans to BIO format if needed.

The extraction prompt is already defined as a reusable template. To use the notebook with new text, change only the `input_text` variable and run the cells.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\\Users\\vasav\\OneDrive\\Documents\\Deep-learning-Lab\\models\\llm")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR

## Terminal Setup Commands

Run these commands in a terminal if you need to check Ollama or download the local model:

```bash
ollama --version
ollama pull qwen2.5:14b-instruct
ollama list
ollama run qwen2.5:14b-instruct
```

If `ollama` is not found, install Ollama separately first, then restart your terminal or notebook kernel.

In [ ]:
import shutil
import subprocess

MODEL_TO_CHECK = "qwen2.5:14b-instruct"


def run_command(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess:
    return subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=timeout,
        check=False,
    )


ollama_path = shutil.which("ollama")
if not ollama_path:
    raise RuntimeError(
        "Ollama command was not found. Install Ollama, restart the terminal or notebook kernel, "
        "then rerun this cell."
    )

print(f"Found Ollama command: {ollama_path}")

version_result = run_command(["ollama", "--version"])
if version_result.returncode == 0:
    print("Ollama version:", version_result.stdout.strip() or version_result.stderr.strip())
else:
    print("Could not read Ollama version.")
    print(version_result.stderr.strip())

list_result = run_command(["ollama", "list"])
if list_result.returncode != 0:
    print("Could not list Ollama models. Make sure the Ollama server is running.")
    print(list_result.stderr.strip())
else:
    model_available = MODEL_TO_CHECK in list_result.stdout
    if model_available:
        print(f"Model is available locally: {MODEL_TO_CHECK}")
    else:
        print(f"Model is not available locally: {MODEL_TO_CHECK}")
        print("Attempting to pull the model. This can take a long time for a 14B model.")
        pull_result = run_command(["ollama", "pull", MODEL_TO_CHECK], timeout=60 * 60)
        if pull_result.returncode != 0:
            raise RuntimeError(
                "Failed to pull qwen2.5:14b-instruct from Ollama. "
                f"stderr: {pull_result.stderr.strip()}"
            )
        print(pull_result.stdout.strip())
        print(f"Model pulled successfully: {MODEL_TO_CHECK}")

In [ ]:
import sys
!"{sys.executable}" -m pip install -U requests pandas spacy jsonschema

import json
import re
import time
import requests
import pandas as pd
import spacy
from jsonschema import validate, ValidationError
from pathlib import Path

In [ ]:
MODEL_NAME = "qwen2.5:14b-instruct"
OLLAMA_URL = "http://localhost:11434/api/generate"

In [ ]:
TAGS_URL = "http://localhost:11434/api/tags"

try:
    tags_response = requests.get(TAGS_URL, timeout=10)
    tags_response.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        "Ollama server is not running. Start Ollama, or run `ollama serve` in a terminal, "
        "then rerun this cell."
    ) from exc

available_models = tags_response.json().get("models", [])
available_model_names = sorted(model.get("name", "") for model in available_models)
print("Ollama server is running.")
print("Available models:")
for name in available_model_names:
    print(f"- {name}")

if MODEL_NAME not in available_model_names:
    print(f"Warning: {MODEL_NAME} was not listed by the Ollama server.")
    print("Run the environment-check cell above or `ollama pull qwen2.5:14b-instruct` in a terminal.")

In [ ]:
ALLOWED_LABELS = [
    "PATIENT",
    "DOCTOR",
    "HOSPITAL",
    "ORGANIZATION",
    "LOCATION",
    "ADDRESS",
    "DATE",
    "TIME",
    "AGE",
    "ID",
    "PHONE",
    "EMAIL",
    "URL",
    "PROFESSION",
    "DISEASE",
    "SYMPTOM",
    "DRUG",
    "DOSAGE",
    "FREQUENCY",
    "ROUTE",
    "DURATION",
    "ANATOMY",
    "PROCEDURE",
    "DIAGNOSTIC_PROCEDURE",
    "LAB_TEST",
    "LAB_VALUE",
    "VITAL_SIGN",
    "CLINICAL_FINDING",
    "MEDICAL_DEVICE",
    "ALLERGY",
    "FAMILY_HISTORY",
    "SOCIAL_HISTORY",
    "TREATMENT",
    "OUTCOME",
]

LABEL_DESCRIPTIONS = {
    "PATIENT": "Patient names or direct patient identifiers as people.",
    "DOCTOR": "Clinician, physician, provider, or care-team member names.",
    "HOSPITAL": "Hospital, clinic, ward, or named care facility.",
    "ORGANIZATION": "Non-hospital organization, company, agency, or institution.",
    "LOCATION": "City, region, country, or non-address location.",
    "ADDRESS": "Street address, postal address, ZIP/postal code, or address-like location.",
    "DATE": "Calendar date or date-like expression.",
    "TIME": "Clock time or time-of-day expression.",
    "AGE": "Patient age or age-like expression.",
    "ID": "Medical record number, patient ID, account ID, visit ID, or other identifier.",
    "PHONE": "Telephone or fax number.",
    "EMAIL": "Email address.",
    "URL": "Web URL or URI.",
    "PROFESSION": "Occupation or professional role.",
    "DISEASE": "Disease, disorder, diagnosis, or chronic condition.",
    "SYMPTOM": "Symptom or patient-reported complaint.",
    "DRUG": "Medication, vaccine, supplement, or drug name.",
    "DOSAGE": "Medication amount, strength, or dose quantity.",
    "FREQUENCY": "Medication frequency or schedule.",
    "ROUTE": "Medication route of administration.",
    "DURATION": "Treatment, medication, symptom, or condition duration.",
    "ANATOMY": "Body part, organ, tissue, or anatomical location.",
    "PROCEDURE": "Therapeutic, surgical, or general clinical procedure.",
    "DIAGNOSTIC_PROCEDURE": "Diagnostic test, imaging, exam, or measurement procedure.",
    "LAB_TEST": "Laboratory test name.",
    "LAB_VALUE": "Laboratory result value including units when present.",
    "VITAL_SIGN": "Vital sign measurement such as blood pressure, pulse, temperature, or oxygen saturation.",
    "CLINICAL_FINDING": "Observed clinical finding that is not clearly a symptom, disease, lab, or vital sign.",
    "MEDICAL_DEVICE": "Medical device, implant, catheter, monitor, or instrument.",
    "ALLERGY": "Allergy or adverse sensitivity.",
    "FAMILY_HISTORY": "Family history statement or inherited-risk mention.",
    "SOCIAL_HISTORY": "Social history such as smoking, alcohol, occupation, or living situation.",
    "TREATMENT": "Treatment plan, therapy, intervention, or recommendation.",
    "OUTCOME": "Clinical outcome, response, disposition, or prognosis.",
}

assert set(ALLOWED_LABELS) == set(LABEL_DESCRIPTIONS), "Every allowed label must have a description."

In [ ]:
JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "text": {"type": "string"},
                    "label": {"type": "string", "enum": ALLOWED_LABELS},
                    "start": {"type": "integer", "minimum": 0},
                    "end": {"type": "integer", "minimum": 0},
                },
                "required": ["text", "label", "start", "end"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["entities"],
    "additionalProperties": False,
}

In [ ]:
def build_prompt(input_text: str) -> str:
    label_block = "\n".join(
        f"- {label}: {LABEL_DESCRIPTIONS[label]}" for label in ALLOWED_LABELS
    )
    schema_block = json.dumps(JSON_SCHEMA, indent=2, ensure_ascii=False)

    return f"""
You are a clinical named entity recognition system.

Extract all clinical and PHI entities from the input clinical text.

Allowed labels and descriptions:
{label_block}

Return only entities that appear exactly in the input text. Use exact character spans from the original input text.
The start offset is inclusive. The end offset is exclusive.
Each entity text must exactly equal input_text[start:end].
Do not hallucinate entities. If unsure, skip the entity.
Do not infer entities that are not explicitly present in the text.

Output must be only valid JSON matching this JSON schema:
{schema_block}

Do not include markdown, explanations, comments, or extra keys.
Do not wrap the JSON in code fences.

Clinical text:
{input_text}
""".strip()

In [ ]:
input_text = """On 03/12/2019 at 09:30, 67-year-old Mr. David Jones, patient ID MRN-45892, was admitted to St. Mary's Hospital in Stuttgart by Dr. Emily Smith, a cardiologist. He reported severe left knee pain, dizziness, nausea, and shortness of breath. His history included osteoarthritis, hypertension, and an allergy to penicillin. An X-ray showed joint-space narrowing, and a blood test showed CRP 18 mg/L and blood pressure 150/95 mmHg. Dr. Smith prescribed ibuprofen 400 mg orally twice daily for 5 days and recommended physiotherapy. The patient can be contacted at [david.jones@example.com](mailto:david.jones@example.com) or +49 711 123456."""

In [ ]:
def call_ollama(prompt: str, model: str = MODEL_NAME, temperature: float = 0.0) -> dict:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": JSON_SCHEMA,
        "options": {
            "temperature": temperature,
            "num_ctx": 8192,
        },
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    except requests.ConnectionError as exc:
        raise RuntimeError(
            "Could not connect to Ollama. Start Ollama, or run `ollama serve` in a terminal, then retry."
        ) from exc
    except requests.Timeout as exc:
        raise RuntimeError("Ollama generation timed out. Retry or increase the request timeout.") from exc
    except requests.RequestException as exc:
        raise RuntimeError(f"Ollama request failed: {exc}") from exc

    if response.status_code != 200:
        raise RuntimeError(
            f"Ollama returned HTTP {response.status_code}: {response.text[:1000]}"
        )

    try:
        response_payload = response.json()
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Ollama returned invalid JSON: {response.text[:1000]}") from exc

    if "response" not in response_payload:
        raise RuntimeError(f"Ollama response is missing the 'response' key: {response_payload}")

    return response_payload

In [ ]:
def parse_model_response(raw_response):
    if isinstance(raw_response, dict) and "response" in raw_response:
        content = raw_response["response"]
    else:
        content = raw_response

    if isinstance(content, str):
        cleaned = content.strip()
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*```$", "", cleaned)
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Model response was not valid JSON: {cleaned[:1000]}") from exc
    else:
        parsed = content

    if isinstance(parsed, list):
        parsed = {"entities": parsed}

    if not isinstance(parsed, dict):
        raise ValueError(f"Model response must be a JSON object or entity list, got {type(parsed).__name__}.")

    if "entities" not in parsed:
        raise ValueError("Model response is missing the required 'entities' key.")

    return {"entities": parsed["entities"]}


def validate_output_schema(output):
    try:
        validate(instance=output, schema=JSON_SCHEMA)
    except ValidationError as exc:
        path = ".".join(str(part) for part in exc.absolute_path)
        location = f" at {path}" if path else ""
        raise ValueError(f"Output JSON failed schema validation{location}: {exc.message}") from exc
    return output

In [ ]:
def find_all_occurrences(source: str, needle: str) -> list[int]:
    if not needle:
        return []

    starts = []
    search_from = 0
    while True:
        idx = source.find(needle, search_from)
        if idx == -1:
            break
        starts.append(idx)
        search_from = idx + 1
    return starts


def repair_offsets(input_text: str, entities: list[dict]) -> list[dict]:
    repaired = []
    seen = set()

    for entity in entities:
        label = str(entity.get("label", "")).strip().upper()
        if label not in ALLOWED_LABELS:
            continue

        text = str(entity.get("text", "")).strip()
        if not text:
            continue

        try:
            original_start = int(entity.get("start", -1))
            original_end = int(entity.get("end", -1))
        except (TypeError, ValueError):
            original_start = -1
            original_end = -1

        start = original_start
        end = original_end

        if 0 <= start <= end <= len(input_text) and input_text[start:end] == text:
            pass
        else:
            matches = find_all_occurrences(input_text, text)
            if not matches:
                continue
            if len(matches) == 1:
                start = matches[0]
            elif 0 <= original_start <= len(input_text):
                start = min(matches, key=lambda idx: abs(idx - original_start))
            else:
                start = matches[0]
            end = start + len(text)

        if input_text[start:end] != text:
            continue

        clean_entity = {
            "text": text,
            "label": label,
            "start": start,
            "end": end,
        }
        key = (clean_entity["text"], clean_entity["label"], clean_entity["start"], clean_entity["end"])
        if key in seen:
            continue
        seen.add(key)
        repaired.append(clean_entity)

    return sorted(repaired, key=lambda item: (item["start"], item["end"]))

In [ ]:
PHI_LABELS = {
    "PATIENT",
    "DOCTOR",
    "HOSPITAL",
    "ORGANIZATION",
    "LOCATION",
    "ADDRESS",
    "DATE",
    "TIME",
    "AGE",
    "ID",
    "PHONE",
    "EMAIL",
    "URL",
    "PROFESSION",
}

CLINICAL_LABELS = set(ALLOWED_LABELS) - PHI_LABELS

LABEL_PRIORITY = {
    "PATIENT": 1,
    "DOCTOR": 1,
    "HOSPITAL": 1,
    "EMAIL": 1,
    "PHONE": 1,
    "ID": 1,
    "ADDRESS": 2,
    "LOCATION": 3,
    "DATE": 4,
    "TIME": 4,
    "AGE": 4,
    "URL": 4,
    "PROFESSION": 5,
    "DRUG": 1,
    "DOSAGE": 1,
    "FREQUENCY": 1,
    "ROUTE": 1,
    "DURATION": 1,
    "VITAL_SIGN": 1,
    "LAB_VALUE": 1,
    "LAB_TEST": 2,
    "ALLERGY": 2,
    "DISEASE": 3,
    "SYMPTOM": 3,
    "ANATOMY": 4,
    "DIAGNOSTIC_PROCEDURE": 4,
    "PROCEDURE": 5,
    "TREATMENT": 5,
    "CLINICAL_FINDING": 6,
    "MEDICAL_DEVICE": 6,
    "FAMILY_HISTORY": 7,
    "SOCIAL_HISTORY": 7,
    "OUTCOME": 7,
}


def spans_overlap(left: dict, right: dict) -> bool:
    return left["start"] < right["end"] and right["start"] < left["end"]


def entity_length(entity: dict) -> int:
    return entity["end"] - entity["start"]


def entity_confidence(entity: dict) -> float:
    try:
        return float(entity.get("confidence", 0.0))
    except (TypeError, ValueError):
        return 0.0


def entity_priority(entity: dict) -> int:
    return LABEL_PRIORITY.get(entity.get("label"), 999)


def choose_same_span(entities: list[dict]) -> dict:
    return sorted(
        entities,
        key=lambda item: (
            entity_priority(item),
            -entity_confidence(item),
            item["label"] not in PHI_LABELS,
            item["label"],
        ),
    )[0]


def choose_overlap_winner(entities: list[dict]) -> dict:
    return sorted(
        entities,
        key=lambda item: (
            -entity_length(item),
            entity_priority(item),
            -entity_confidence(item),
            item["start"],
            item["end"],
        ),
    )[0]


def resolve_overlaps(entities: list[dict]) -> list[dict]:
    by_span = {}
    for entity in entities:
        by_span.setdefault((entity["start"], entity["end"]), []).append(entity)

    span_deduped = [choose_same_span(items) for items in by_span.values()]
    sorted_entities = sorted(
        span_deduped,
        key=lambda item: (
            item["start"],
            -entity_length(item),
            -entity_confidence(item),
            entity_priority(item),
        ),
    )

    selected = []
    for entity in sorted_entities:
        overlapping = [item for item in selected if spans_overlap(item, entity)]
        if not overlapping:
            selected.append(entity)
            continue

        winner = choose_overlap_winner(overlapping + [entity])
        selected = [item for item in selected if item not in overlapping]
        if winner not in selected:
            selected.append(winner)

    clean_selected = [
        {
            "text": item["text"],
            "label": item["label"],
            "start": int(item["start"]),
            "end": int(item["end"]),
        }
        for item in selected
    ]
    return sorted(clean_selected, key=lambda item: (item["start"], item["end"]))

In [ ]:
def extract_entities(input_text: str) -> dict:
    if not isinstance(input_text, str) or not input_text.strip():
        raise ValueError("input_text must be a non-empty string.")

    prompt = build_prompt(input_text)
    raw_response = call_ollama(prompt)
    model_output = parse_model_response(raw_response)
    validate_output_schema(model_output)

    repaired_entities = repair_offsets(input_text, model_output["entities"])
    clean_entities = resolve_overlaps(repaired_entities)

    for entity in clean_entities:
        if input_text[entity["start"]:entity["end"]] != entity["text"]:
            raise ValueError(f"Invalid final span after repair: {entity}")

    validate_output_schema({"entities": clean_entities})
    return {
        "text": input_text,
        "entities": clean_entities,
    }

In [ ]:
result = extract_entities(input_text)
result

In [ ]:
print(json.dumps(result["entities"], indent=2, ensure_ascii=False))

In [ ]:
entities_df = pd.DataFrame(result["entities"])
entities_df

In [ ]:
output_path = PROJECT_DIR / "qwen25_clinical_ner_output.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

output_path

In [ ]:
def spans_to_bio(input_text: str, entities: list[dict]) -> pd.DataFrame:
    nlp = spacy.blank("en")
    doc = nlp(input_text)

    rows = []
    sorted_entities = sorted(entities, key=lambda item: (item["start"], item["end"]))

    for token in doc:
        token_start = token.idx
        token_end = token.idx + len(token.text)
        label = "O"

        for entity in sorted_entities:
            if token_start >= entity["start"] and token_end <= entity["end"]:
                prefix = "B" if token_start == entity["start"] else "I"
                label = f"{prefix}-{entity['label']}"
                break

        rows.append(
            {
                "token": token.text,
                "start": token_start,
                "end": token_end,
                "label": label,
            }
        )

    return pd.DataFrame(rows)


bio_df = spans_to_bio(input_text, result["entities"])
display(bio_df)

bio_output_path = PROJECT_DIR / "qwen25_clinical_ner_bio.csv"
bio_df.to_csv(bio_output_path, index=False)
bio_output_path

In [ ]:
def run_clinical_ner(input_text: str):
    result = extract_entities(input_text)
    print(json.dumps(result["entities"], indent=2, ensure_ascii=False))
    return result


run_clinical_ner(input_text)

## Usage

Change only `input_text`, then run all cells. The final JSON entity list is available in `result["entities"]`. The full JSON output is saved to `qwen25_clinical_ner_output.json`, and the optional BIO CSV is saved to `qwen25_clinical_ner_bio.csv`.

Ollama must be running locally before extraction. This workflow is designed for local inference through Ollama; no real patient data should be sent to external APIs.